In [0]:
import mlflow

from pyspark.ml import PipelineModel

from src.training.train_fault_classifier import (
    prepare_training_data
)

from src.monitoring.prediction_monitor import (
    calculate_prediction_distribution
)

In [0]:
SOURCE_TABLE = (
    "tep_anomaly.served.training_base"
)

source_df = spark.table(
    SOURCE_TABLE
)

prepared_df = prepare_training_data(
    source_df
)

print(
    f"Rows loaded: {prepared_df.count():,}"
)

In [0]:
# get current champion model

from mlflow import MlflowClient

MODEL_NAME = (
    "tep_anomaly.served.tep_fault_classifier"
)

client = MlflowClient()

champion = (
    client.get_model_version_by_alias(
        MODEL_NAME,
        "Champion"
    )
)

print(
    f"Champion Version: "
    f"{champion.version}"
)

print(
    f"Champion Run: "
    f"{champion.run_id}"
)

In [0]:
# load champion model

from mlflow import MlflowClient

MODEL_NAME = (
    "tep_anomaly.served.tep_fault_classifier"
)

client = MlflowClient()

champion = (
    client.get_model_version_by_alias(
        MODEL_NAME,
        "Champion"
    )
)

print(
    f"Champion Version: "
    f"{champion.version}"
)

print(
    f"Champion Run: "
    f"{champion.run_id}"
)

In [0]:
import mlflow

model_uri = (
    f"models:/{MODEL_NAME}@Champion"
)

print(
    f"Loading model from: {model_uri}"
)

model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/tep_anomaly/served/mlflow_artifacts"
)

print("Model loaded successfully")

In [0]:
# generate predictions 

predictions_df = model.transform(
    prepared_df
)

display(
    predictions_df.select(
        "prediction"
    )
)

In [0]:
# calculate prediction distribution 

distribution_df = (
    calculate_prediction_distribution(
        predictions_df
    )
)

display(
    distribution_df
)

In [0]:
# persist monitoring results 

from pyspark.sql import functions as F

(
    distribution_df
    .withColumn(
        "monitoring_timestamp",
        F.current_timestamp()
    )
    .withColumnRenamed(
        "prediction",
        "prediction_class"
    )
    .withColumnRenamed(
        "count",
        "prediction_count"
    )
    .write
    .mode("append")
    .saveAsTable(
        "tep_anomaly.served.prediction_monitoring_history"
    )
)

In [0]:
# validate results

display(
    spark.sql(
        """
        SELECT *
        FROM tep_anomaly.served.prediction_monitoring_history
        ORDER BY monitoring_timestamp DESC
        """
    )
)